# Native HDF5 reader demo

This hardware-independent notebook uses the current `hdf5_store` public API.
It does not require a QICK connection and does not use the removed `data_archive` module.


In [ ]:
from pathlib import Path
import sys
import h5py
import matplotlib.pyplot as plt
import numpy as np

cwd = Path.cwd().resolve()
REPO_ROOT = next((p for p in (cwd, *cwd.parents) if (p / 'QickworkspaceV2').is_dir()), cwd)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from QickworkspaceV2.tools import find_experiments, inspect_file, load_result, validate_file

DATA_ROOT = REPO_ROOT / 'tutorial' / 'read_tool' / 'data'
records = find_experiments(data_root=DATA_ROOT)
print('Indexed experiments:', len(records))


## SQLite-backed discovery


In [ ]:
[(r.timestamp_local, r.experiment_type, r.qubits, r.tags, r.comment_preview) for r in records[:12]]


In [ ]:
history = find_experiments(
    experiment_type='s008_T1_ge', qubit='Q1', tags=['repeated'],
    start='2026-07-08', end='2026-07-14', data_root=DATA_ROOT,
)
[(r.timestamp_local, r.session_id, r.quality, r.comment_preview) for r in history]


## Load complete results only when needed


In [ ]:
trend = []
for ref in reversed(history):
    result = load_result(ref.path)
    trend.append((ref.timestamp_local, result.fit_result['T1_us'][0]))

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot([x for x, _ in trend], [y for _, y in trend], 'o-')
ax.tick_params(axis='x', rotation=45)
ax.set(title='Repeated Q1 T1 measurements', ylabel='T1 (us)', xlabel='Local timestamp')
fig.tight_layout()


In [ ]:
t1_ref = history[0]
info = inspect_file(t1_ref.path)
report = validate_file(t1_ref.path)
print('Valid:', report.valid)
info


## Selective raw HDF5 access with h5py


In [ ]:
with h5py.File(t1_ref.path, 'r') as h5:
    root = h5['metagroup'] if 'metagroup' in h5 else h5
    iq = np.asarray(root['raw/iq'])[:10]
    dims = root['raw/iq'].attrs.get('dims', '[]')
    axes = {name: np.asarray(group['values']) for name, group in root['axes'].items()}

print('IQ shape:', iq.shape)
print('Stored dims:', dims)
print('Axes:', {name: values.shape for name, values in axes.items()})
iq[:3]


## Single-shot data


In [ ]:
single_shot_refs = find_experiments(
    experiment_type='s010_single_shot_ge', qubit='Q2', limit=1, data_root=DATA_ROOT,
)
single_shot = load_result(single_shot_refs[0].path)
shots = np.asarray(single_shot.raw_iq)
print('Single-shot shape:', shots.shape)
shots.reshape(-1)[:3]


## Plot stored raw IQ


In [ ]:
for ref in records[:8]:
    result = load_result(ref.path)
    values = np.squeeze(np.asarray(result.raw_iq))
    fig, ax = plt.subplots(figsize=(6, 3))
    if values.ndim == 1:
        x = result.x_axis if result.x_axis is not None and len(result.x_axis) == values.size else np.arange(values.size)
        ax.plot(x, np.abs(values))
    else:
        image = np.abs(values)
        while image.ndim > 2:
            image = image.mean(axis=-1)
        ax.imshow(image, origin='lower', aspect='auto')
    ax.set_title(ref.experiment_type)
    plt.show()
    plt.close(fig)


## Full result


In [ ]:
result = load_result(t1_ref.path)
result.experiment_id, result.comment, result.fit_result, np.asarray(result.raw_iq).shape
